In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 1. import numpy, pandas, matplotlib, seaborn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 2. matplotlib inline 설정
%matplotlib inline

In [4]:
data_processed_parquet_path = "drive/MyDrive/Semiconductor-AnomalyDetection/data/processed/master_df.parquet"
master_df = pd.read_parquet(data_processed_parquet_path)

In [5]:
#Baseline goal
# 현재 데이터로 예측 가능한가?
# 가장 기본 모델로 어느 정도 성능이 나오나?
# 이후 RF, XGBoost 같은 더 복잡한 모델이 baseline보다 좋아지는가?

# master_df 불러오기
# → X(입력), y(정답) 나누기
# → train/test 나누기
# → train으로 전처리+학습
# → test로 예측
# → 성능 평가

In [6]:
import os
import json # 파일 읽고 쓰는 library
import joblib # 모델 load/save library


from sklearn.model_selection import train_test_split #data split to train and test
from sklearn.pipeline import Pipeline #여러 단계 묶기
from sklearn.impute import SimpleImputer # 결측치 채우는 도구
from sklearn.preprocessing import StandardScaler #Feature Scaling mean:0, var:1 값 범위 맞추기
from sklearn.linear_model import LogisticRegression #baseline model
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)


In [7]:
#1. setting
base_dir = "drive/MyDrive/Semiconductor-AnomalyDetection"

#models dir생성
models_dir = os.path.join(base_dir, "models")
os.makedirs(models_dir, exist_ok=True)
print("created:", models_dir)


models_baseline_dir = os.path.join(base_dir, "models", "baseline")
result_dir = os.path.join(base_dir, "results", "baseline")

os.makedirs(models_baseline_dir, exist_ok=True)
os.makedirs(result_dir, exist_ok=True)


created: drive/MyDrive/Semiconductor-AnomalyDetection/models


In [8]:
#2. data loading
data_path = os.path.join(base_dir, "data", "processed", "master_df.parquet")
# DATA_PATH = "drive/MyDrive/Semiconductor-AnomalyDetection/data/processed/master_df.parquet"

df = pd.read_parquet(data_path)
print(f"Loaded data shape: {df.shape}")


target_col = "label"
drop_cols = ["sample_id", "timestamp"]

# 필요 없는 컬럼 제거
# if drop_cols exist, put it in the list
drop_cols_exist = [col for col in drop_cols if col in df.columns]
df = df.drop(columns=drop_cols_exist, errors="ignore")

# 타겟 존재 확인
if target_col not in df.columns:
    raise ValueError(f"'{target_col}' column not found in dataframe.")

# X, Y 분리
X = df.drop(columns=[target_col])
y = df[target_col].copy()

print("Target distribution:")
print(y.value_counts(dropna=False))

Loaded data shape: (1567, 593)
Target distribution:
label
-1    1463
 1     104
Name: count, dtype: int64


In [9]:


RANDOM_STATE = 42
TEST_SIZE = 0.2
#3. checking targets
# 만약 현재 라벨이 -1, 1 이라면 그대로 logistic regression 가능
# 다만 평가할 때 positive class를 1로 둘 거라면 아래처럼 정리해두면 좋음.
unique_labels = sorted(y.dropna().unique().tolist())
print("unique labels:", unique_labels)

# {1, -1}

if y.isna().sum() > 0:
    raise ValueError("Target column contains missing values. Handle target NaNs first.")


#4. Only numeric values will be used
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
X = X[numeric_cols]

print(f"Number of numeric features: {len(numeric_cols)}")

#5.train and test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

#Pipeline
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000,
        class_weight="balanced"   # 불균형 데이터면 baseline에서 유용
    ))
])


#7. Train
pipeline.fit(X_train, y_train)

#8.prediction
y_pred = pipeline.predict(X_test)

# roc_auc용 probability
# logistic regression은 predict_proba 사용 가능
y_proba = pipeline.predict_proba(X_test)[:, 1] if len(np.unique(y_train)) == 2 else None

#9.Evaluation
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
    "recall": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
    "f1": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
}

if y_proba is not None:
    metrics["roc_auc"] = roc_auc_score(y_test, y_proba)

Confusion_Matrix = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, zero_division=0)

print("\n=== Metrics ===")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

print("\n=== Confusion Matrix ===")
print(Confusion_Matrix)

print("\n=== Classification Report ===")
print(report)


#10.Store result


# with open(os.path.join(result_dir, "logistic_metrics.json"), "w") as f:
#     json.dump(metrics, f, indent=4)

# with open(os.path.join(result_dir, "classification_report.txt"), "w") as f:
#     f.write(report)

# np.savetxt(
#     os.path.join(result_dir, "confusion_matrix.txt"),
#     Confusion_Matrix,
#     fmt="%d"
# )

# joblib.dump(pipeline, os.path.join(models_baseline_dir, "logistic_baseline.pkl"))

# print("\nSaved metrics, report, confusion matrix, and model.")

unique labels: [-1, 1]
Number of numeric features: 590
Train shape: (1253, 590)
Test shape : (314, 590)

=== Metrics ===
accuracy: 0.8439
precision: 0.1111
recall: 0.1905
f1: 0.1404
roc_auc: 0.6438

=== Confusion Matrix ===
[[261  32]
 [ 17   4]]

=== Classification Report ===
              precision    recall  f1-score   support

          -1       0.94      0.89      0.91       293
           1       0.11      0.19      0.14        21

    accuracy                           0.84       314
   macro avg       0.52      0.54      0.53       314
weighted avg       0.88      0.84      0.86       314

